# Stage 2: Local embeddings and similarity

This notebook turns text chunks into **embedding vectors** using a local Ollama model (`nomic-embed-text`), then ranks chunks for a question with **cosine similarity**.

Still no Chroma, LangChain, or chat LLM — only embeddings + simple math.

## What is an embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text.

- Similar meanings → vectors that point in similar directions
- Unrelated meanings → vectors that point in different directions

In RAG we embed each **chunk**, embed the **user question**, then pick the chunks whose vectors are closest to the question. That is retrieval — before the LLM answers.

**Prerequisite:** Ollama is running and `nomic-embed-text` is installed:

```bash
ollama serve          # if the server is not already running
ollama pull nomic-embed-text
```

## Load and chunk the knowledge base (from Stage 1)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "knowledge_base").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"
assert KB_DIR.is_dir(), f"Missing knowledge base at {KB_DIR}"


def load_text_documents(folder: Path) -> list[dict]:
    documents = []
    for path in sorted(folder.glob("*.txt")):
        documents.append({"source": path.name, "text": path.read_text(encoding="utf-8").strip()})
    return documents


def chunk_by_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def build_chunk_records(documents: list[dict]) -> list[dict]:
    records = []
    for doc in documents:
        for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
            records.append({"source": doc["source"], "chunk_id": i, "text": chunk})
    return records


chunks = build_chunk_records(load_text_documents(KB_DIR))
print(f"Loaded {len(chunks)} paragraph chunks from {KB_DIR}")
print("Example:", chunks[0])

## Call Ollama for embeddings

We use Ollama's HTTP API (`/api/embeddings`) with `requests` — no LangChain.

In [ ]:
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
EMBED_MODEL = "nomic-embed-text"


def check_ollama() -> None:
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        r.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            "Cannot reach Ollama at 127.0.0.1:11434. Start it with: ollama serve"
        ) from exc
    names = {m["name"] for m in r.json().get("models", [])}
    # Ollama may list "nomic-embed-text" or "nomic-embed-text:latest"
    if not any(n.startswith(EMBED_MODEL) for n in names):
        raise RuntimeError(
            f"Model {EMBED_MODEL!r} not found. Install with: ollama pull {EMBED_MODEL}"
        )
    print("Ollama OK. Models:", ", ".join(sorted(names)))


def embed_text(text: str) -> list[float]:
    """Return an embedding vector for one string via Ollama."""
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": text},
        timeout=120,
    )
    response.raise_for_status()
    vector = response.json()["embedding"]
    if not vector:
        raise RuntimeError("Ollama returned an empty embedding")
    return vector


check_ollama()
demo = embed_text("How long do refunds take?")
print(f"Demo embedding length (dimensions): {len(demo)}")
print(f"First 5 numbers: {demo[:5]}")

## Embed every chunk

This may take a short while the first time (model load + one request per chunk).

In [ ]:
embedded_chunks = []
for i, chunk in enumerate(chunks):
    vector = embed_text(chunk["text"])
    embedded_chunks.append({**chunk, "embedding": vector})
    print(f"[{i + 1}/{len(chunks)}] {chunk['source']}#{chunk['chunk_id']} → {len(vector)} dims")

print(f"\nEmbedded {len(embedded_chunks)} chunks")

## Cosine similarity

**Cosine similarity** measures how aligned two vectors are (about -1 to 1 for typical embeddings; for many embedding models scores are between 0 and 1).

Higher score ⇒ more similar meaning.

In [ ]:
import numpy as np


def cosine_similarity(a: list[float], b: list[float]) -> float:
    va = np.asarray(a, dtype=np.float64)
    vb = np.asarray(b, dtype=np.float64)
    denom = np.linalg.norm(va) * np.linalg.norm(vb)
    if denom == 0:
        return 0.0
    return float(np.dot(va, vb) / denom)


def rank_chunks(question: str, records: list[dict], top_k: int = 3) -> list[tuple[float, dict]]:
    q_vec = embed_text(question)
    scored = [
        (cosine_similarity(q_vec, rec["embedding"]), rec)
        for rec in records
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top_k]


def show_ranking(question: str, top_k: int = 3) -> None:
    print(f"Question: {question}\n")
    for rank, (score, rec) in enumerate(rank_chunks(question, embedded_chunks, top_k=top_k), start=1):
        preview = rec["text"].replace("\n", " ")
        if len(preview) > 140:
            preview = preview[:140] + "..."
        print(f"{rank}. score={score:.3f}  {rec['source']}#{rec['chunk_id']}")
        print(f"   {preview}\n")

## Demo: in-scope question (should match refund / return text)

In [ ]:
show_ranking("How many days does a refund take to appear on my statement?")

## Demo: another in-scope question (shipping)

In [ ]:
show_ranking("How long is express delivery after an order ships?")

## Demo: out-of-scope question

The top scores should look weaker / less clearly related. Later stages will use a threshold (or the LLM) to say "I don't know" when the docs do not help.

In [ ]:
show_ranking("What is the capital of France?")

in_scope = rank_chunks("How many days does a refund take to appear on my statement?", embedded_chunks, top_k=1)[0][0]
out_of_scope = rank_chunks("What is the capital of France?", embedded_chunks, top_k=1)[0][0]
print(f"Best score (refund question): {in_scope:.3f}")
print(f"Best score (unrelated question): {out_of_scope:.3f}")
if in_scope > out_of_scope:
    print("OK: in-scope question ranked higher than the unrelated question.")
else:
    print("Unexpected: unrelated question scored as high or higher — inspect chunks and model.")

## What you should notice

- Embeddings turn text into fixed-length number lists.
- Cosine similarity ranks which chunks are closest to a question.
- Relevant policy text rises to the top; unrelated questions get weaker matches.

## Next stage (not built yet)

**Stage 3:** store these vectors in **Chroma** (local vector database) and retrieve top-k chunks with a short API instead of scoring every chunk by hand.